# Extract tables from pdf

In [1]:
import pdfplumber
import pandas as pd

PDF_PATH = r"D:\PC DISAINE\toky\Perso-D\red\momo\transaction-history-20260627_111819321.pdf"  # change to your file path

all_tables = []

with pdfplumber.open(PDF_PATH) as pdf:
    for i, page in enumerate(pdf.pages):
        table = page.extract_table()
        if table:
            print(f"Page {i+1}: found table with {len(table)-1} rows")
            all_tables.append(table)

# Use first table found, or merge all
raw = all_tables[0]
df = pd.DataFrame(raw[1:], columns=raw[0])  # row 0 = headers
df.head()

Page 1: found table with 12 rows
Page 2: found table with 16 rows
Page 3: found table with 17 rows
Page 4: found table with 17 rows
Page 5: found table with 17 rows
Page 6: found table with 18 rows
Page 7: found table with 16 rows
Page 8: found table with 17 rows
Page 9: found table with 17 rows
Page 10: found table with 16 rows
Page 11: found table with 17 rows
Page 12: found table with 17 rows
Page 13: found table with 17 rows
Page 14: found table with 18 rows
Page 15: found table with 16 rows
Page 16: found table with 16 rows
Page 17: found table with 17 rows
Page 18: found table with 16 rows
Page 19: found table with 17 rows
Page 20: found table with 16 rows
Page 21: found table with 17 rows
Page 22: found table with 16 rows
Page 23: found table with 2 rows


,Transaction\nId,Status,Transaction Type,Date,From,To,Amount (RWF),Fee\n(RWF),Balance\nAfter\n(RWF)
0,28805974985,SUCCESSFUL,PAYMENT,2026-06-26 22:27:07,250795962609\nJoie Livine NYINAWUMUNTU,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,200,N/A,4602695
1,28805848805,SUCCESSFUL,PAYMENT,2026-06-26 22:15:53,250788693771\nGeorges KWIZERA,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,3000,N/A,4602495
2,28805464934,SUCCESSFUL,PAYMENT,2026-06-26 21:48:09,250780745466\nEmery MANZI,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,3000,N/A,4599510
3,28805412570,SUCCESSFUL,PAYMENT,2026-06-26 21:44:28,250780489816\nEmelinne IGIHOZO,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,5000,N/A,4596525
4,28805109859,SUCCESSFUL,PAYMENT,2026-06-26 21:26:35,250781613870\nPatricia Narindra Raoel BOTO,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,5000,N/A,4591550


# Clean raw tabular data

In [2]:
headers = all_tables[0][0]
frames = []

for t in all_tables:
    rows = [row for row in t if row != headers]  # drop repeated header rows
    frames.append(pd.DataFrame(rows, columns=headers))

df = pd.concat(frames, ignore_index=True)

df = df.replace("", pd.NA)
df = df.dropna(how="any")

df = df.rename(columns={'Amount\n(RWF)': 'Amount (RWF)'})

df['Amount (RWF)'] = df['Amount (RWF)'].astype(int)

df.shape

(350, 9)

In [10]:
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d %H:%M:%S")
start = pd.to_datetime("2026-06-26 00:00:00")
end   = pd.to_datetime("2026-06-26 23:59:59")

df_subset = df[(df["Date"] >= start) & (df["Date"] <= end)]
df_subset.shape

(57, 9)

# Calculate the sum of all transactions

In [11]:
# Assign +/- sign based on the type column
df_subset["Signed Amount"] = df_subset["Amount (RWF)"].where(df_subset["Transaction Type"] == "PAYMENT", -df_subset["Amount (RWF)"])

# Then sum
total_transactions = df_subset["Signed Amount"].sum()
print(total_transactions)

536300


## Total fees

In [12]:
balance_2026_06_18 = 4068990
balance_2026_06_19 = 4602695
total_actual_transactions = balance_2026_06_19 - balance_2026_06_18
print(total_actual_transactions)

533705


In [13]:
error = 0
total_fees = total_transactions - total_actual_transactions - error

print(total_fees)

2595


# Check individual transaction fee

In [14]:
df_subset = df_subset.rename(columns={'Balance\nAfter\n(RWF)': 'Balance'})
df_subset["Balance"] = df_subset["Balance"].astype("int")

In [15]:
df_subset["transaction_fee"] = df_subset["Amount (RWF)"] - ( df_subset["Balance"] - df_subset["Balance"].shift(-1) )

In [16]:
df_subset

,Transaction\nId,Status,Transaction Type,Date,From,To,Amount (RWF),Fee\n(RWF),Balance,Signed Amount,transaction_fee
0,28805974985,SUCCESSFUL,PAYMENT,2026-06-26 22:27:07,250795962609\nJoie Livine NYINAWUMUNTU,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,200,N/A,4602695,200,0.0
1,28805848805,SUCCESSFUL,PAYMENT,2026-06-26 22:15:53,250788693771\nGeorges KWIZERA,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,3000,N/A,4602495,3000,15.0
2,28805464934,SUCCESSFUL,PAYMENT,2026-06-26 21:48:09,250780745466\nEmery MANZI,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,3000,N/A,4599510,3000,15.0
3,28805412570,SUCCESSFUL,PAYMENT,2026-06-26 21:44:28,250780489816\nEmelinne IGIHOZO,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,5000,N/A,4596525,5000,25.0
4,28805109859,SUCCESSFUL,PAYMENT,2026-06-26 21:26:35,250781613870\nPatricia Narindra Raoel BOTO,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,5000,N/A,4591550,5000,25.0
5,28805096927,SUCCESSFUL,PAYMENT,2026-06-26 21:26:12,250792881437\nANDRIAMIHAJA FENOHASINA\nFLORENT,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,3500,N/A,4586575,3500,17.0
6,28804457707,SUCCESSFUL,PAYMENT,2026-06-26 20:54:32,250788500369\nAline UMULISA,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,2500,N/A,4583092,2500,12.0
7,28804149878,SUCCESSFUL,PAYMENT,2026-06-26 20:41:50,250788316029\nThadée TWAGIRAYEZU,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,2500,N/A,4580604,2500,12.0
8,28804083841,SUCCESSFUL,PAYMENT,2026-06-26 20:39:27,250788316029\nThadée TWAGIRAYEZU,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,15000,N/A,4578116,15000,75.0
9,28803760946,SUCCESSFUL,PAYMENT,2026-06-26 20:27:04,250798654587\nLEIGH JOYFAVOUR\nBLESSING,38998332\nRED IKIRWA\nFLAVOURS Ltd RED\nIKIRWA...,5000,N/A,4563191,5000,25.0
